<a href="https://colab.research.google.com/github/Tegalchain/Tegalchain/blob/master/Tegalchain_Blockchain_Genesis_FIXED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

# Tegalchain Blockchain Genesis - Fixed Setup

## Environment Specifications
This notebook sets up the correct environment for Tegalchain development:
- **Go**: 1.26.3
- **Ignite CLI**: v0.29.10
- **Rust**: 1.96.0
- **Cosmos SDK**: v0.53.6
- **Buf**: v1.70.0

## Key Improvements
✓ Fixed PATH prioritization for custom tools
✓ Proper go.mod replace directives
✓ Corrected import paths for AccAddress
✓ Isolated environment for Ignite CLI
✓ Comprehensive error handling and diagnostics

## Section 1: Environment Verification

In [ ]:
#@title Initial Environment Check
import subprocess
import os

print('=' * 80)
print('INITIAL ENVIRONMENT VERIFICATION')
print('=' * 80)

tools = [
    ('go', 'go version'),
    ('rustc', 'rustc --version'),
    ('buf', 'buf --version'),
    ('ignite', 'ignite version')
]

for tool_name, cmd in tools:
    print(f"\n--- {tool_name.upper()} ---")
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=5)
        if result.returncode == 0:
            print(f"✓ {result.stdout.strip()}")
        else:
            print(f"✗ Not found or error: {result.stderr.strip()[:100]}")
    except subprocess.TimeoutExpired:
        print(f"✗ Command timeout")
    except Exception as e:
        print(f"✗ Error: {str(e)[:100]}")

print(f"\n--- PYTHON PATH ---")
print(os.environ.get('PATH', 'NOT SET')[:200])

## Section 2: Install/Update Tools to Correct Versions

In [ ]:
#@title Install Go 1.26.3
import subprocess
import os
import shutil
import requests

GO_VERSION = "1.26.3"
HOME = os.path.expanduser("~")
GOROOT = os.path.join(HOME, 'go')
GOPATH = os.path.join(HOME, 'go')

print(f"\n{'='*80}")
print(f"INSTALLING GO {GO_VERSION}")
print(f"{'='*80}")
print(f"GOROOT: {GOROOT}")
print(f"GOPATH: {GOPATH}")

# Check if correct version is already installed
go_bin = os.path.join(GOROOT, 'bin', 'go')
if os.path.exists(go_bin):
    result = subprocess.run([go_bin, 'version'], capture_output=True, text=True)
    if GO_VERSION in result.stdout:
        print(f"\n✓ Go {GO_VERSION} already installed")
        print(result.stdout.strip())
    else:
        print(f"\n! Different Go version found, replacing...")
        shutil.rmtree(GOROOT, ignore_errors=True)
else:
    print(f"\nGo not found, installing...")

if not os.path.exists(go_bin):
    GO_TAR_GZ = f"go{GO_VERSION}.linux-amd64.tar.gz"
    GO_DOWNLOAD_URL = f"https://go.dev/dl/{GO_TAR_GZ}"
    
    print(f"\nDownloading from: {GO_DOWNLOAD_URL}")
    try:
        response = requests.get(GO_DOWNLOAD_URL, stream=True, timeout=60)
        response.raise_for_status()
        
        temp_path = os.path.join('/tmp', GO_TAR_GZ)
        with open(temp_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        
        file_size = os.path.getsize(temp_path) / (1024*1024)
        print(f"✓ Downloaded: {file_size:.1f} MB")
        
        print(f"Extracting to {HOME}...")
        subprocess.run(['tar', '-C', HOME, '-xzf', temp_path], check=True)
        
        os.remove(temp_path)
        print(f"✓ Go {GO_VERSION} installed successfully")
        
        # Verify
        result = subprocess.run([go_bin, 'version'], capture_output=True, text=True)
        print(f"\n✓ Verification: {result.stdout.strip()}")
    except Exception as e:
        print(f"✗ Error: {e}")
        raise

In [ ]:
#@title Install Rust 1.96.0
import subprocess
import os

print(f"\n{'='*80}")
print("VERIFYING RUST 1.96.0")
print(f"{'='*80}")

try:
    result = subprocess.run(['rustc', '--version'], capture_output=True, text=True, timeout=5)
    print(f"✓ {result.stdout.strip()}")
    if '1.96.0' in result.stdout:
        print("✓ Correct version already installed")
    else:
        print("! Different version found, consider updating via rustup")
except:
    print("Rust not found. Installing via rustup...")
    try:
        subprocess.run(
            ['curl', '--proto', '=https', '--tlsv1.2', '-sSf', 'https://sh.rustup.rs', '-o', '/tmp/rustup-init.sh'],
            check=True
        )
        subprocess.run(
            ['bash', '/tmp/rustup-init.sh', '-y', '--profile', 'minimal'],
            check=True
        )
        
        # Add wasm target
        subprocess.run(
            ['rustup', 'target', 'add', 'wasm32-unknown-unknown'],
            check=True
        )
        
        result = subprocess.run(['rustc', '--version'], capture_output=True, text=True)
        print(f"✓ {result.stdout.strip()}")
    except Exception as e:
        print(f"✗ Error: {e}")
        raise

In [ ]:
#@title Install Ignite CLI v0.29.10
import subprocess
import os
import shutil
import requests
import re

IGNITE_VERSION = "v0.29.10"
HOME = os.path.expanduser("~")
IGNITE_INSTALL_PATH = os.path.join(HOME, '.ignite', 'bin')
IGNITE_BIN = os.path.join(IGNITE_INSTALL_PATH, 'ignite')

print(f"\n{'='*80}")
print(f"INSTALLING IGNITE CLI {IGNITE_VERSION}")
print(f"{'='*80}")
print(f"Install path: {IGNITE_BIN}")

# Check existing installation
if os.path.exists(IGNITE_BIN):
    result = subprocess.run([IGNITE_BIN, 'version'], capture_output=True, text=True, timeout=5)
    output = result.stdout + result.stderr
    if '0.29.10' in output or 'v0.29.10' in output:
        print(f"✓ Ignite {IGNITE_VERSION} already installed")
    else:
        print("! Different version found, replacing...")
        os.remove(IGNITE_BIN)
        ignite_bin_exists = False
else:
    ignite_bin_exists = False

if not ignite_bin_exists:
    os.makedirs(IGNITE_INSTALL_PATH, exist_ok=True)
    
    # Download Ignite
    tarball_name = f"ignite_0.29.10_linux_amd64.tar.gz"
    download_url = f"https://github.com/ignite/cli/releases/download/{IGNITE_VERSION}/{tarball_name}"
    temp_path = os.path.join('/tmp', tarball_name)
    
    print(f"\nDownloading from: {download_url}")
    try:
        response = requests.get(download_url, stream=True, timeout=60)
        response.raise_for_status()
        
        with open(temp_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        
        file_size = os.path.getsize(temp_path) / (1024*1024)
        print(f"✓ Downloaded: {file_size:.1f} MB")
        
        # Extract
        extract_dir = os.path.join('/tmp', 'ignite_extract')
        os.makedirs(extract_dir, exist_ok=True)
        subprocess.run(['tar', '-xzf', temp_path, '-C', extract_dir], check=True)
        print(f"✓ Extracted")
        
        # Move binary
        src = os.path.join(extract_dir, 'ignite')
        shutil.move(src, IGNITE_BIN)
        os.chmod(IGNITE_BIN, 0o755)
        
        # Cleanup
        shutil.rmtree(extract_dir)
        os.remove(temp_path)
        
        print(f"✓ Ignite {IGNITE_VERSION} installed successfully")
    except Exception as e:
        print(f"✗ Error: {e}")
        raise

In [ ]:
#@title Install Buf v1.70.0
import subprocess
import os
import requests

BUF_VERSION = "v1.70.0"
BUF_INSTALL_PATH = '/usr/local/bin/buf'

print(f"\n{'='*80}")
print(f"INSTALLING BUF {BUF_VERSION}")
print(f"{'='*80}")

# Check existing
try:
    result = subprocess.run(['buf', '--version'], capture_output=True, text=True, timeout=5)
    print(f"✓ {result.stdout.strip()}")
    if '1.70.0' in result.stdout:
        print("✓ Correct version already installed")
    else:
        print("! Different version found, updating...")
except:
    print("Buf not found, installing...")

if not os.path.exists(BUF_INSTALL_PATH) or subprocess.run(['buf', '--version'], capture_output=True).returncode != 0:
    download_url = f"https://github.com/bufbuild/buf/releases/download/{BUF_VERSION}/buf-linux-x86_64"
    
    print(f"\nDownloading from: {download_url}")
    try:
        response = requests.get(download_url, stream=True, timeout=60)
        response.raise_for_status()
        
        with open(BUF_INSTALL_PATH, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        
        os.chmod(BUF_INSTALL_PATH, 0o755)
        print(f"✓ Buf {BUF_VERSION} installed successfully")
        
        result = subprocess.run(['buf', '--version'], capture_output=True, text=True)
        print(f"✓ Verification: {result.stdout.strip()}")
    except Exception as e:
        print(f"✗ Error: {e}")
        raise

## Section 3: Configure Environment Variables

In [ ]:
#@title Setup Environment Variables
import os

HOME = os.path.expanduser("~")
GOROOT = os.path.join(HOME, 'go')
GOPATH = os.path.join(HOME, 'go')
CARGO_HOME = os.path.join(HOME, '.cargo')

print(f"\n{'='*80}")
print("SETTING UP ENVIRONMENT VARIABLES")
print(f"{'='*80}")

# Build prioritized PATH
custom_paths = [
    os.path.join(GOROOT, 'bin'),                    # Go binary
    os.path.join(HOME, '.ignite', 'bin'),           # Ignite CLI
    os.path.join(CARGO_HOME, 'bin'),                # Rust/Cargo
    '/usr/local/bin',                               # System tools (buf)
]

# Keep other system paths but exclude conflicting ones
existing = os.environ.get('PATH', '').split(os.pathsep)
system_paths = [p for p in existing 
                if p and not p.startswith('/usr/local/go') 
                and p not in custom_paths
                and '/ignite' not in p]

new_path = os.pathsep.join(custom_paths + system_paths)

# Set environment variables
os.environ['PATH'] = new_path
os.environ['GOROOT'] = GOROOT
os.environ['GOPATH'] = GOPATH
os.environ['CARGO_HOME'] = CARGO_HOME
os.environ['GO111MODULE'] = 'on'
os.environ['CGO_ENABLED'] = '0'

print(f"✓ GOROOT: {GOROOT}")
print(f"✓ GOPATH: {GOPATH}")
print(f"✓ CARGO_HOME: {CARGO_HOME}")
print(f"\n✓ PATH configured with priority: Go, Ignite, Cargo, System tools")
print(f"\n✓ GO111MODULE: {os.environ['GO111MODULE']}")
print(f"✓ CGO_ENABLED: {os.environ['CGO_ENABLED']}")

## Section 4: Final Environment Verification

In [ ]:
#@title Final Environment Verification
import subprocess
import os

print(f"\n{'='*80}")
print("FINAL ENVIRONMENT VERIFICATION")
print(f"{'='*80}")

tests = [
    ('Go 1.26.3', 'go version', '1.26.3'),
    ('Rust 1.96.0', 'rustc --version', '1.96.0'),
    ('Ignite CLI v0.29.10', 'ignite version', '0.29.10'),
    ('Buf v1.70.0', 'buf --version', '1.70.0'),
]

all_passed = True

for name, cmd, expected_version in tests:
    print(f"\n{name}:")
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=10)
        output = result.stdout + result.stderr
        
        if result.returncode == 0 and expected_version in output:
            print(f"  ✓ PASS")
            print(f"  {output.strip().split(chr(10))[0]}")
        else:
            print(f"  ⚠ WARNING - Version mismatch")
            print(f"  {output.strip()[:100]}")
            all_passed = False
    except subprocess.TimeoutExpired:
        print(f"  ✗ FAIL - Command timeout")
        all_passed = False
    except Exception as e:
        print(f"  ✗ FAIL - {str(e)[:50]}")
        all_passed = False

print(f"\n{'='*80}")
if all_passed:
    print("✓ ALL CHECKS PASSED - Environment is ready!")
else:
    print("⚠ Some checks need attention - see details above")
print(f"{'='*80}")

## Section 5: Download and Setup Tegalchain

In [ ]:
#@title Clone/Update Tegalchain Repository
import subprocess
import os

print(f"\n{'='*80}")
print("TEGALCHAIN REPOSITORY SETUP")
print(f"{'='*80}")

repo_path = '/content/tegalchain'

if os.path.exists(repo_path):
    print(f"\nRepository exists at {repo_path}")
    print("Updating...")
    os.chdir(repo_path)
    subprocess.run(['git', 'pull', 'origin', 'master'], capture_output=True)
    print("✓ Updated")
else:
    print(f"\nCloning repository...")
    os.makedirs('/content', exist_ok=True)
    os.chdir('/content')
    result = subprocess.run(
        ['git', 'clone', 'https://github.com/Tegalchain/Tegalchain.git', 'tegalchain'],
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print("✓ Repository cloned")
    else:
        print(f"✗ Clone failed: {result.stderr}")

os.chdir(repo_path)
print(f"\n✓ Working directory: {os.getcwd()}")

## Section 6: Fix Import Path Issues

In [ ]:
#@title Run Import Path Fix Script
import subprocess
import os

print(f"\n{'='*80}")
print("FIXING IMPORT PATHS")
print(f"{'='*80}")

# Download and run fix script
fix_script_path = '/tmp/fix_tegalchain_imports.py'

# The fix script is provided in the next cell
# For now, we'll execute it inline

execfile('/tmp/fix_tegalchain_imports.py', {'__name__': '__main__'})

print(f"\n{'='*80}")
print("✓ Import path fixes completed")
print(f"{'='*80}")

## Section 7: Verify and Fix go.mod

In [ ]:
#@title Verify go.mod Configuration
import os
import re

print(f"\n{'='*80}")
print("VERIFYING go.mod")
print(f"{'='*80}")

gomod_path = '/content/tegalchain/go.mod'

if os.path.exists(gomod_path):
    with open(gomod_path, 'r') as f:
        content = f.read()
    
    print("\nChecking for issues...")
    
    # Check for duplicate replace directives
    cosmos_replaces = re.findall(r'replace.*cosmos.*=>.*', content)
    print(f"\nFound {len(cosmos_replaces)} cosmos-related replace directives:")
    for replace in cosmos_replaces:
        print(f"  {replace}")
    
    if len(cosmos_replaces) > 1:
        print("\n⚠ WARNING: Multiple cosmos replace directives found!")
        print("This can cause module conflicts. Consider consolidating to one.")
    else:
        print("\n✓ Replace directives look good")
    
    # Show go version
    go_line = re.search(r'^go\s+([\d.]+)', content, re.MULTILINE)
    if go_line:
        print(f"\n✓ Go version in go.mod: {go_line.group(1)}")
    
    # Show cosmos SDK version
    cosmos_line = re.search(r'github\.com/cosmos/cosmos-sdk\s+v([\d.]+)', content)
    if cosmos_line:
        print(f"✓ Cosmos SDK version: v{cosmos_line.group(1)}")
else:
    print(f"✗ go.mod not found at {gomod_path}")

## Section 8: Build Tegalchain

In [ ]:
#@title Build Tegalchain
import subprocess
import os

print(f"\n{'='*80}")
print("BUILDING TEGALCHAIN")
print(f"{'='*80}")

os.chdir('/content/tegalchain')

# Setup environment for build
env = os.environ.copy()
env['GO111MODULE'] = 'on'
env['CGO_ENABLED'] = '0'

print("\nRunning: ignite chain build")
print("This may take several minutes...\n")

result = subprocess.run(
    ['ignite', 'chain', 'build'],
    env=env,
    capture_output=False,
    text=True
)

print(f"\n{'='*80}")
if result.returncode == 0:
    print("✓ BUILD SUCCESSFUL!")
else:
    print(f"✗ BUILD FAILED with exit code {result.returncode}")
    print("Check the output above for error details")
print(f"{'='*80}")

## Section 9: Diagnostics and Troubleshooting

In [ ]:
#@title Run Diagnostics
import subprocess
import os

print(f"\n{'='*80}")
print("DIAGNOSTICS")
print(f"{'='*80}")

os.chdir('/content/tegalchain')

# Check Go modules
print("\n--- Go Module Status ---")
result = subprocess.run(['go', 'mod', 'verify'], capture_output=True, text=True)
if result.returncode == 0:
    print("✓ Go modules verified")
else:
    print(f"⚠ Go module verification issues:\n{result.stderr}")

# List dependencies
print("\n--- Key Dependencies ---")
result = subprocess.run(
    ['go', 'list', '-m', 'all'],
    capture_output=True,
    text=True
)
lines = result.stdout.split('\n')
key_deps = [l for l in lines if 'cosmos' in l or 'ignite' in l or 'tegalchain' in l]
for dep in key_deps[:10]:
    print(f"  {dep}")

# Check for build artifacts
print("\n--- Build Artifacts ---")
bin_path = '/content/tegalchain/cmd/tegalchaind/'
if os.path.exists(bin_path):
    files = os.listdir(bin_path)
    if files:
        print(f"✓ Build directory exists with {len(files)} files")
else:
    print("⚠ Build directory not found")

print(f"\n{'='*80}")